# 01 — Understanding the data, and cleaning it

## What this project is about

Softwood Pvt Ltd is a textile **sourcing company** based in Pakistan. It does not
own a factory and does not manufacture anything. It sits in the middle of two
groups:

- On one side, Pakistani mills that make fabric and garments like Rajby, Samad
  Rubber, Interloop, Pak Denim, and about a hundred others.
- On the other side, European clothing brands that want to buy like Takko,
  Tiffosi, Jennyfer, Vilanova, Minoti, and others.

This company manages the order, and takes a **commission** on
the value of the deal. That commission is the company's entire income.

The company shared six years of its deal records with us. This notebook is the
first step: understand exactly what is in that file, find what is wrong with it,
and produce a clean version we can trust.

## What one row means

The file has one row per deal. Read a row like a short story:

> On 11 June 2019 a deal was signed. A Pakistani mill called Pak Denim made
> 6,392 units of a cotton-polyester denim fabric for a European buyer called
> Tiffosi. Each unit cost \$2.07, so the deal was worth \$13,232. Softwood took
> a 2% commission, which came to \$264. The goods shipped on 8 January 2020 and
> the buyer had 30 days to pay.

There are about 4,700 such stories in the file, from June 2019 to April 2025.

## What this notebook does

1. Look at the raw file as it actually is
2. Find every problem in it, with evidence
3. Fix what can be fixed, flag what cannot, and record every decision
4. Save a clean dataset the rest of the project can build on


In [1]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

raw = pd.read_csv("../data/raw/textile_sales.csv")
raw.columns = [c.strip() for c in raw.columns]

print(f"Rows:    {len(raw):,}")
print(f"Columns: {raw.shape[1]}")
print(f"Period:  {pd.to_datetime(raw.Contract_Date).min():%b %Y} "
      f"to {pd.to_datetime(raw.Contract_Date).max():%b %Y}")

Rows:    4,725
Columns: 26
Period:  Jun 2019 to Apr 2025


In [2]:
raw.head(3)

,Team_Head,Supplier,Customer,Contract_Date,Dispatch_Date,Payment_Days,Expected_Payment,Style_Name,Rate,Dollar_Exchange_Rate,Dollar_Rate,Contract_#,Contract_Qty.,Shipped_Quantity,Bal,Sales_Volume,Exchange,Commission_Dollar_Value,Commission_Percentage,Commission_PKR_Value,Commission_SW_Percentage,SW_Commission_PKR,Tax_%,SW_After_Tax_Value,Received,SW_Rcvd_after_Tax
0,SWU,Pak Denim,Tiffosi Fab,2019-06-11 00:00:00,2020-01-08 00:00:00,30,2020-02-07 00:00:00,"3023, 10.75Oz, 59/60"", 3/1 RHT, Cotton Poly",325,0.0064,2.07,PDM/19-2208,6392.0,6392.0,0.0,13232,157.00,264.64,0.02,41548.0,0.008,16619,0.1,14957.0,37393,14957.3
1,Bilal,Rajby,Takko,2019-07-01 00:00:00,2019-12-21 00:00:00,115,2020-04-14 00:00:00,TKMEN,5.5,1.0000,5.50,2055253/0,17300.0,17300.0,0.0,95150,166.00,4757.50,0.05,789745.0,0.020,315898,0.05,300103.0,750258,300103.0
2,Bilal,Rajby,Takko,2019-07-01 00:00:00,2020-01-22 00:00:00,115,2020-05-16 00:00:00,TKMEN,5.55,1.0000,5.55,2055253/5,7624.0,7624.0,0.0,42313,166.01,2115.66,0.05,351221.0,0.020,140488,0.05,133464.0,333660,133463.9


## The 26 columns, grouped by what they tell us

Twenty-six columns looks like a lot, but they fall into five simple groups.

| Group | Columns | What it tells us |
|---|---|---|
| Who | `Team_Head`, `Supplier`, `Customer` | Who handled the deal, who made the goods, who bought them |
| What | `Style_Name`, `Contract_Qty.`, `Shipped_Quantity`, `Bal` | Which fabric, how much was ordered, how much actually shipped |
| When | `Contract_Date`, `Dispatch_Date`, `Payment_Days`, `Expected_Payment` | Signed, shipped, and when payment is due |
| How much | `Rate`, `Dollar_Rate`, `Sales_Volume`, `Exchange`, `Dollar_Exchange_Rate` | Unit price, total deal value, and the USD/PKR rate that day |
| What we earned | the 10 remaining commission and tax columns | Softwood's cut, before and after tax |

The last group is where the money is, so it is worth looking at closely.


In [3]:
money_cols = ["Sales_Volume", "Commission_Percentage", "Commission_Dollar_Value",
              "Exchange", "Commission_PKR_Value", "Tax_%", "Received",
              "Commission_SW_Percentage", "SW_Commission_PKR", "SW_Rcvd_after_Tax"]
raw[money_cols].head(3)

,Sales_Volume,Commission_Percentage,Commission_Dollar_Value,Exchange,Commission_PKR_Value,Tax_%,Received,Commission_SW_Percentage,SW_Commission_PKR,SW_Rcvd_after_Tax
0,13232,0.02,264.64,157.00,41548.0,0.1,37393,0.008,16619,14957.3
1,95150,0.05,4757.50,166.00,789745.0,0.05,750258,0.020,315898,300103.0
2,42313,0.05,2115.66,166.01,351221.0,0.05,333660,0.020,140488,133463.9


### These ten columns are really one number and nine calculations

Look at the first row above and follow the arithmetic:

```
deal value                $13,232      (Sales_Volume)
x commission rate           2%         (Commission_Percentage)   <-- the only real decision
= commission              $264.64      (Commission_Dollar_Value)
x USD/PKR rate              157
= commission            Rs 41,548      (Commission_PKR_Value)
- tax                      10%
= net commission        Rs 37,393      (Received)
```

Only one of these is a business decision: **the commission rate**. Everything
else follows automatically once the rate is set. Let us check whether that is
true for the whole file, not just this one row.


In [4]:
def as_number(s):
    # Spreadsheet text -> numbers. Handles commas, $ signs and '-' placeholders.
    return pd.to_numeric(
        s.astype(str).str.replace(r"[,$\s]", "", regex=True)
         .replace({"-": np.nan, "": np.nan, "nan": np.nan}),
        errors="coerce")


def agrees(calculated, stated, tolerance=0.01):
    # Share of rows where a formula reproduces the stated value.
    usable = calculated.notna() & stated.notna() & (stated.abs() > 1e-9)
    return ((calculated[usable] - stated[usable]).abs() / stated[usable].abs() < tolerance).mean()


value = as_number(raw.Sales_Volume)
commission = as_number(raw.Commission_Dollar_Value)
rate = as_number(raw.Commission_Percentage)
net_pkr = as_number(raw.Received)

checks = {
    "deal value  = quantity x unit price": agrees(raw["Contract_Qty."] * raw.Dollar_Rate, value),
    "commission  = deal value x rate":      agrees(value * rate, commission),
    "commission in PKR = commission x FX":  agrees(commission * raw.Exchange, as_number(raw.Commission_PKR_Value)),
    "net = commission in PKR - tax":        agrees(as_number(raw.Commission_PKR_Value) * (1 - as_number(raw["Tax_%"])), net_pkr),
}
for label, share in checks.items():
    print(f"{share:6.1%}  of rows follow:  {label}")

 99.3%  of rows follow:  deal value  = quantity x unit price
 98.2%  of rows follow:  commission  = deal value x rate
 99.9%  of rows follow:  commission in PKR = commission x FX
 95.9%  of rows follow:  net = commission in PKR - tax


**This confirms it.** The formulas hold on almost every row. So of the ten
money columns, nine are bookkeeping. The business only decides one thing per
deal: what commission rate to charge.

This matters more than it looks, and we come back to it in notebook 04. An
earlier version of this project built a machine learning model to *predict*
commission — while giving the model the after-tax commission as an input. That
is like predicting someone's age after being shown their ID card. We will show
that properly later.

For now the useful conclusion is: **the commission rate is the thing worth
studying.**

---

## Problem 1: numbers stored as text

Excel writes a dash into empty accounting cells. When pandas reads that, the
whole column becomes text, and text cannot be added up or averaged.


In [5]:
text_but_numeric = ["Sales_Volume", "SW_Commission_PKR", "Tax_%", "Received",
                    "Payment_Days", "Rate"]
report = []
for col in text_but_numeric:
    values = raw[col].astype(str)
    unparseable = values[as_number(values).isna()]
    report.append({
        "column": col,
        "stored as": raw[col].dtype.name,
        "bad cells": len(unparseable),
        "example": unparseable.iloc[0] if len(unparseable) else "-",
    })
pd.DataFrame(report)

,column,stored as,bad cells,example
0,Sales_Volume,str,51,$ -
1,SW_Commission_PKR,str,59,-
2,Tax_%,str,1,-
3,Received,str,2,-
4,Payment_Days,str,1,1900-01-15 00:00:00
5,Rate,str,132,Rs 437


Six columns are affected. Four of them carry Excel's `-` / `$ -` placeholder for
an empty accounting cell; `Rate` is the worst, with 132 rows written as text like
`Rs685`; and one `Payment_Days` cell holds a date where a number belongs.

The counts are small, but until this is fixed, anything that sums or averages
these columns is quietly wrong — and `Rate` is a price column.

---

## Problem 2: the same company written several different ways

If "Tiffosi Fab" and "Tiffosi fab" are treated as two different buyers, then
every per-customer number is split in half. We looked for names that differ only
by capitalisation or an obvious typo.


In [6]:
import difflib

def near_duplicates(names, cutoff=0.82):
    unique, seen, pairs = sorted(set(names)), set(), []
    for a in unique:
        if a in seen:
            continue
        similar = [b for b in unique
                   if b != a and difflib.SequenceMatcher(None, a.lower(), b.lower()).ratio() > cutoff]
        if similar:
            counts = names.value_counts()
            pairs.append({"name": f"{a} ({counts[a]})",
                          "looks like": ", ".join(f"{s} ({counts[s]})" for s in similar)})
            seen.update(similar + [a])
    return pd.DataFrame(pairs)

print("CUSTOMERS")
display(near_duplicates(raw.Customer))
print("SUPPLIERS")
display(near_duplicates(raw.Supplier))

CUSTOMERS


,name,looks like
0,Authentic Style Denim (59),Authentic Style knits (251)
1,Bizbee (1),Bizzbee (45)
2,CDRL Fab (1),CRDL Fab (2)
3,Gerry Weber (4),Gerry Weber-Fab (2)
4,Get Over (26),Get over (3)
5,Jomo Fashion (1),Jomo Fashion Fab (2)
6,Reporter Young (54),"Reporter Young Fab (4), Reporter Young Knit (66)"
7,Tiffosi Fab (203),"Tiffosi fab (24), Tifossi Fab (1)"
8,Tiffosi Knit (10),Tiffosi knit (3)
9,Top Secret (2),Top Secret Fab (1)


SUPPLIERS


,name,looks like
0,AIM Textile (5),Haram Textile (124)
1,AR Apparels (208),"Amna Apparels (57), Aruj Apparel (1), Pak Appa..."
2,Ashar Textile (2),Haram Textile (124)
3,Indigo (84),indigo (2)
4,JB industries (19),MR Industries (150)
5,Safa Tex (5),Safa tex (6)
6,Shorts Quality (30),Shorts quality (2)
7,Zahra Industries (30),MR Industries (150)


Two different things are showing up in that list, and telling them apart matters.

**Real duplicates — same company, typed differently.** These get merged:

- `Tiffosi fab` / `Tifossi Fab` -> `Tiffosi Fab`
- `Tiffosi knit` -> `Tiffosi Knit`, `Get over` -> `Get Over`
- `Bizbee` -> `Bizzbee`, `CDRL Fab` -> `CRDL Fab`
- `indigo` -> `Indigo`, `Safa tex` -> `Safa Tex`, `Shorts quality` -> `Shorts Quality`

**Not duplicates — genuinely different records.** `Takko` and `Takko Fab` are
*not* a typo of each other. `Authentic Style Denim` and `Authentic Style knits`
are not either. Merging these would destroy real information, and the next
section explains why.

---

## Problem 3: there is a hidden column inside the customer name

Notice the pattern: `Takko`, `Takko Fab`, `Takko Knit`. Same brand, different
suffix. The suffix is not decoration — it says which **product line** the deal
belongs to. Fabric is a different business from finished garments.

If that is true, the commission rate should behave differently across the
suffixes. Let us check.


In [7]:
import re

SUFFIX = re.compile(r"[\s\-]+(fab|fabric|knit|knits|denim|woven|garment|garments|apparel)s?$", re.I)

work = raw.copy()
work["rate"] = as_number(work.Commission_Percentage)
work = work[(work.rate > 0) & (work.rate < 1)]
work["division"] = work.Customer.str.extract(SUFFIX, expand=False).str.lower().fillna("garment")

summary = work.groupby("division").rate.agg(
    deals="size", p10=lambda s: s.quantile(.10), median="median",
    p90=lambda s: s.quantile(.90))
summary["at exactly 2%"] = work.assign(two=work.rate.round(4).eq(0.02)).groupby("division").two.mean()
summary.round(4)

,deals,p10,median,p90,at exactly 2%
division,,,,,
denim,71,0.0200,0.0200,0.0466,0.7746
fab,738,0.0200,0.0200,0.0200,0.9892
garment,2988,0.0200,0.0345,0.0600,0.2838
knit,872,0.0248,0.0500,0.0921,0.0298


**This is the most important finding in the notebook.**

Deals in the **fab** (fabric) line are at exactly 2.00% commission in **98.9%**
of cases. The 10th and 90th percentile are both 2.00% — there is no spread at
all. It is a fixed contractual rate.

Deals in the **knit** (garment) line are almost never at 2%. Their median is 5%
and they spread widely, because each one is negotiated.

So Softwood is really running two different businesses with two different
pricing models, and the only place that is recorded is a suffix inside a text
column. Let us confirm it holds inside a single brand, where nothing else
changes.


In [8]:
takko = work[work.Customer.str.lower().str.startswith("takko")]
takko.groupby("Customer").rate.agg(
    deals="size", median="median",
    share_at_2pct=lambda s: s.round(4).eq(0.02).mean()).round(3)

,deals,median,share_at_2pct
Customer,,,
Takko,817,0.05,0.009
Takko Fab,439,0.02,0.989
Takko Knit,487,0.05,0.039
Takko socks,12,0.08,0.000


Same brand, same buyer relationship — and `Takko Fab` sits at 2% while `Takko`
and `Takko Knit` sit at 5%. The product line, not the customer, sets the rate.

Every rate comparison from here on has to respect this split. Comparing a fabric
deal against a garment deal is comparing two unrelated things. We therefore
split `Customer` into two proper columns: `customer_brand` and
`product_division`.

---

## Problem 4: the Team_Head column holds two different kinds of thing

`Team_Head` should be the Softwood staff member who handled the deal. Most
values are first names. But some are company names.


In [9]:
mills = set(raw.Supplier.unique())
looks_like_company = raw.Team_Head.isin(mills)

print(f"Rows where Team_Head is a known mill name: {looks_like_company.sum()}")
print(f"Of those, Team_Head equals the Supplier on the same row: "
      f"{(raw.Team_Head == raw.Supplier)[looks_like_company].sum()}")
print()
# The individual staff names are replaced with labels here. They are personal
# data about identifiable employees, they are not needed for the argument, and
# this notebook is published. The deal counts are what matter and are unchanged.
people = raw.Team_Head[~looks_like_company].value_counts()
people.index = [f"staff_{i+1}" for i in range(len(people))]

print("Value counts:")
pd.DataFrame({"staff (anonymised)": people.head(8)}).join(
    pd.DataFrame({"company names": raw.Team_Head[looks_like_company].value_counts().head(8)}),
    how="outer").fillna("")

Rows where Team_Head is a known mill name: 354
Of those, Team_Head equals the Supplier on the same row: 0

Value counts:


,staff (anonymised),company names
Active Apparel,,11.0
Azgard9,,7.0
Cotton Web,,4.0
Eastern,,20.0
Interloop,,36.0
Rajby,,19.0
SWU,,242.0
Shafi,,6.0
staff_1,1532.0,
staff_2,1163.0,


It is not a simple copy-paste error: in **none** of those rows does `Team_Head`
match the `Supplier` on the same row. So we cannot say the column was
accidentally filled from the supplier column, and we cannot recover who the
actual staff member was.

The honest response is to label the column rather than guess or delete. We add
`team_head_type` with values `person` and `organization`, keep the original
value, and let each later analysis decide whether to include organizations.

One note for context: `SWU` accounts for most of the company-name rows and is
almost certainly Softwood itself, since the company's own initials appear
throughout the commission columns. We do not rely on that assumption anywhere.

The staff names are shown as `staff_1 … staff_n` above. They are personal data
about identifiable employees, the analysis never uses them, and this notebook is
public. The company names are kept because the whole point of this section is
that they are the wrong kind of value for the column.

---

## Problem 5: rows that break business rules

A few rows describe things that cannot happen.


In [10]:
contract_on = pd.to_datetime(raw.Contract_Date, errors="coerce")
dispatch_on = pd.to_datetime(raw.Dispatch_Date, errors="coerce")
lead = (dispatch_on - contract_on).dt.days

rules = {
    "commission rate is zero":                 (as_number(raw.Commission_Percentage) == 0),
    "commission rate is 100% or more":          (as_number(raw.Commission_Percentage) >= 1),
    "contract quantity is zero or negative":    (raw["Contract_Qty."] <= 0),
    "USD/PKR rate outside 100-350":             ~raw.Exchange.between(100, 350),
    "shipped more than was ordered":            (raw.Shipped_Quantity > raw["Contract_Qty."]),
    "goods dispatched before contract signed":  (lead < 0),
    "lead time longer than two years":          (lead > 730),
}
pd.DataFrame([{"rule broken": k, "rows": int(v.sum()),
               "share": f"{v.mean():.2%}"} for k, v in rules.items()])

,rule broken,rows,share
0,commission rate is zero,12,0.25%
1,commission rate is 100% or more,33,0.70%
2,contract quantity is zero or negative,76,1.61%
3,USD/PKR rate outside 100-350,13,0.28%
4,shipped more than was ordered,39,0.83%
5,goods dispatched before contract signed,16,0.34%
6,lead time longer than two years,4,0.08%


These split into two kinds.

**Unusable — the row is dropped.** A deal with a zero or 100%+ commission rate,
or zero quantity, has nothing to analyse. An impossible exchange rate makes
every derived figure wrong. Together this is about 2.9% of the file.

**Usable but suspicious — the row is kept and flagged.** Shipping slightly more
than ordered is normal in textiles (mills over-run). A dispatch date before the
contract date is a typing error in the date, but the money on that row is still
valid — so we keep the row, blank out the lead time, and flag it.

The principle: never delete a row for being inconvenient. Delete it only when it
cannot answer the question, and count every deletion.

---

## Running the cleaner

All of the above is implemented in `src/clean_data.py`, so it runs the same way
every time and nothing is done by hand.


In [11]:
from clean_data import run

result = run("../data/raw/textile_sales.csv",
             "../data/processed/deals_clean.csv",
             "../reports/data_quality_audit.csv")

result.audit_frame

,step,rule,rows,action
0,load,rows in raw export,4725,kept
1,types,payment_days: text placeholders could not be p...,1,set to missing
2,types,unit_rate_pkr: text placeholders could not be ...,132,set to missing
3,types,deal_value_usd: text placeholders could not be...,51,set to missing
4,types,sw_commission_pkr: text placeholders could not...,59,set to missing
5,types,tax_pct: text placeholders could not be parsed,1,set to missing
6,types,commission_after_tax_pkr: text placeholders co...,2,set to missing
7,names,supplier: case / spelling variants merged,10,renamed to majority spelling
8,names,customer: case / spelling variants merged,33,renamed to majority spelling
9,structure,"team_head values that are company names, not s...",372,"labelled, kept"


Every row of that table is a decision, with a count and a reason. Anyone can
check our work against it.

## What we ended up with


In [12]:
deals = result.data
kept = len(deals) / len(raw)

print(f"Raw rows:    {len(raw):,}")
print(f"Clean rows:  {len(deals):,}   ({kept:.1%} of the original kept)")
print(f"Period:      {deals.contract_date.min():%b %Y} to {deals.contract_date.max():%b %Y}")
print(f"Suppliers:   {deals.supplier.nunique()}")
print(f"Customers:   {deals.customer.nunique()}  ({deals.customer_brand.nunique()} brands x product lines)")
print(f"Fabrics:     {deals.style_name.nunique():,} distinct style names")
print()
print("Deals by product line:")
print(deals.product_division.value_counts().to_string())
print()
print("Commission rate, after cleaning:")
print(deals.commission_pct.describe([.05, .25, .5, .75, .95]).round(4).to_string())

Raw rows:    4,725
Clean rows:  4,590   (97.1% of the original kept)
Period:      Jun 2019 to Apr 2025
Suppliers:   100
Customers:   73  (53 brands x product lines)
Fabrics:     2,401 distinct style names

Deals by product line:
product_division
garment    2931
knit        871
fab         730
denim        58

Commission rate, after cleaning:
count    4590.0000
mean        0.0395
std         0.0263
min         0.0001
5%          0.0200
25%         0.0200
50%         0.0300
75%         0.0500
95%         0.0826
max         0.3333


The commission rate now tops out at 33% instead of 100%, and the impossible
values are gone.

## Where this leaves us

We started with a finance export and turned it into a dataset we can reason
about. Along the way we learned three things that shape the whole project:

1. **Nine of the ten money columns are formulas.** The only real decision in the
   data is the commission rate.
2. **The business has two pricing models, not one.** Fabric deals are fixed at
   2%; garment deals are negotiated around 5%. This was hidden in a text suffix.
3. **The commission rate varies a lot inside the negotiated side.** That is
   where the money is being won or lost.

Next, notebook 02 looks at what the business actually did over these six years —
and finds the problem the rest of the project exists to solve.
